###SECTION C

###C1


In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1378")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DateType, IntegerType, DoubleType

def standardize_str(col):
    trimmed = F.trim(col)
    return F.when(trimmed == "", F.lit(None)).otherwise(F.upper(trimmed))

df_bronze_orders = spark.read.table(f"{catalog}.bronze.orders")

allowed_status = ["CANCELLED", "DELIVERED", "PLACED"]
df_standardized = df_bronze_orders.select(
    standardize_str(F.col("order_id")).alias("order_id"),
    standardize_str(F.col("customer_id")).alias("customer_id"),
    standardize_str(F.col("store_id")).alias("store_id"),
    F.when(standardize_str(F.col("order_status")) == "CANCELED", "CANCELLED")
     .otherwise(standardize_str(F.col("order_status")))
     .alias("order_status"),
    standardize_str(F.col("payment.method")).alias("payment_method"),
    F.lit("UNKNOWN").alias("channel"),
    F.expr("try_to_timestamp(order_ts)").alias("order_ts"),
    F.expr("try_to_timestamp(updated_at)").alias("updated_at"),
    F.expr("""
        transform(items, (x, i) -> named_struct(
            'line_number', cast(i + 1 as int),
            'product_id', CASE WHEN trim(x.product_id) = '' THEN NULL ELSE upper(trim(x.product_id)) END,
            'quantity', try_cast(x.qty as int),
            'unit_price', try_cast(x.unit_price as double),
            'discount_amount', coalesce(try_cast(x.discount as double), 0.0)
        ))
    """).alias("items")
).withColumn(
    "order_date", F.to_date(F.col("order_ts"))
).where(F.col("order_status").isin(allowed_status))


###C2,C3

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# C2: De-duplication — keep latest version per order_id
window = Window.partitionBy('order_id').orderBy(F.col('updated_at').desc())
df_latest = df_standardized.withColumn('row_num', F.row_number().over(window)).filter(F.col('row_num') == 1).drop('row_num')

# C3: Validation & Quarantine
# Collect reference IDs as Python sets (serializable, unlike Spark DataFrames)
customer_ids = set([row["customer_id"] for row in spark.read.table(f"{catalog}.silver.customers").select("customer_id").distinct().collect()])
store_ids = set([row["store_id"] for row in spark.read.table(f"{catalog}.silver.stores").select("store_id").distinct().collect()])
product_ids = set([row["product_id"] for row in spark.read.table(f"{catalog}.silver.products").select("product_id").distinct().collect()])

def get_reject_reason(customer_id, store_id, order_ts, items):
    if customer_id is None:
        return 'MISSING_CUSTOMER'
    if customer_id not in customer_ids:
        return 'UNKNOWN_CUSTOMER'
    if store_id is None or store_id not in store_ids:
        return 'UNKNOWN_STORE'
    if order_ts is None:
        return 'INVALID_TIMESTAMP'
    if not items or len(items) == 0:
        return 'EMPTY_ORDER'
    for item in items:
        if item.product_id is None or item.product_id not in product_ids:
            return 'UNKNOWN_PRODUCT'
        if item.quantity is None or item.quantity <= 0:
            return 'INVALID_QUANTITY'
        if item.unit_price is None or item.unit_price <= 0:
            return 'INVALID_PRICE'
    return None

reject_reason_udf = udf(get_reject_reason, StringType())

df_validated = df_latest.withColumn(
    'reject_reason',
    reject_reason_udf('customer_id', 'store_id', 'order_ts', 'items')
)

valid_orders = df_validated.filter(F.col('reject_reason').isNull()).drop('reject_reason')
quarantine_orders = df_validated.filter(F.col('reject_reason').isNotNull())

# C5: MERGE into silver.orders (idempotent incremental maintenance)
from delta.tables import DeltaTable

orders_table_name = f"{catalog}.silver.orders"
if not spark.catalog.tableExists(orders_table_name):
    valid_orders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(orders_table_name)
else:
    orders_table = DeltaTable.forName(spark, orders_table_name)
    (orders_table.alias("target")
        .merge(valid_orders.alias("source"), "target.order_id = source.order_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    # Remove orders now in quarantine (never in both silver.orders and silver.quarantine_orders)
    (orders_table.alias("target")
        .merge(quarantine_orders.select("order_id").alias("source"), "target.order_id = source.order_id")
        .whenMatchedDelete()
        .execute())

###C4


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable

order_items = valid_orders.selectExpr(
    "order_id",
    "explode(items) as item",
    "updated_at"
).select(
    "order_id",
    F.col("item.line_number").cast("int").alias("line_number"),
    F.col("item.product_id").alias("product_id"),
    F.col("item.quantity").cast("int").alias("quantity"),
    F.col("item.unit_price").cast(DecimalType(10,2)).alias("unit_price"),
    F.col("item.discount_amount").cast(DecimalType(10,2)).alias("discount_amount"),
    (F.col("item.quantity") * F.col("item.unit_price")).cast(DecimalType(12,2)).alias("gross_amount"),
    ((F.col("item.quantity") * F.col("item.unit_price") - F.col("item.discount_amount")).cast(DecimalType(12,2))).alias("net_amount"),
    "updated_at"
)
# C5: MERGE into silver.order_items (idempotent incremental maintenance)
order_items_table_name = f"{catalog}.silver.order_items"
if not spark.catalog.tableExists(order_items_table_name):
    order_items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(order_items_table_name)
else:
    order_items_table = DeltaTable.forName(spark, order_items_table_name)
    (order_items_table.alias("target")
        .merge(order_items.alias("source"), "target.order_id = source.order_id AND target.line_number = source.line_number")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    # Remove items for orders now in quarantine
    (order_items_table.alias("target")
        .merge(quarantine_orders.select("order_id").alias("source"), "target.order_id = source.order_id")
        .whenMatchedDelete()
        .execute())

quarantine_orders_out = quarantine_orders.select(
    "order_id",
    "reject_reason",
    "updated_at",
    F.to_json(F.struct([F.col(c) for c in quarantine_orders.columns])).alias("raw_record"),
    F.current_timestamp().alias("quarantined_at")
)
# C5: MERGE into silver.quarantine_orders (idempotent — quarantined_at set only on insert)
quarantine_table_name = f"{catalog}.silver.quarantine_orders"
if not spark.catalog.tableExists(quarantine_table_name):
    quarantine_orders_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(quarantine_table_name)
else:
    quarantine_table = DeltaTable.forName(spark, quarantine_table_name)
    (quarantine_table.alias("target")
        .merge(quarantine_orders_out.alias("source"), "target.order_id = source.order_id")
        .whenMatchedUpdate(set={
            "reject_reason": "source.reject_reason",
            "updated_at": "source.updated_at",
            "raw_record": "source.raw_record"
        })
        .whenNotMatchedInsert(values={
            "order_id": "source.order_id",
            "reject_reason": "source.reject_reason",
            "updated_at": "source.updated_at",
            "raw_record": "source.raw_record",
            "quarantined_at": "source.quarantined_at"
        })
        .execute())
    # Remove quarantine entries for orders now valid
    (quarantine_table.alias("target")
        .merge(valid_orders.select("order_id").alias("source"), "target.order_id = source.order_id")
        .whenMatchedDelete()
        .execute())


###C5

In [0]:
# C5: Idempotency verification — re-merge same data, confirm no changes
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Capture state before re-merge
orders_count_before = spark.table(f"{catalog}.silver.orders").count()
items_count_before = spark.table(f"{catalog}.silver.order_items").count()
quarantine_count_before = spark.table(f"{catalog}.silver.quarantine_orders").count()
orders_hash_before = spark.sql(f"SELECT hash(*) FROM {catalog}.silver.orders ORDER BY order_id").collect()
items_hash_before = spark.sql(f"SELECT hash(*) FROM {catalog}.silver.order_items ORDER BY order_id, line_number").collect()
quarantine_pairs_before = spark.sql(f"SELECT order_id, reject_reason FROM {catalog}.silver.quarantine_orders ORDER BY order_id").collect()

# Re-merge silver.orders with same valid_orders (no new bronze data)
orders_table = DeltaTable.forName(spark, f"{catalog}.silver.orders")
(orders_table.alias("target")
    .merge(valid_orders.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

# Re-merge silver.order_items with same data
order_items = valid_orders.selectExpr("order_id", "explode(items) as item", "updated_at").select(
    "order_id",
    F.col("item.line_number").cast("int").alias("line_number"),
    F.col("item.product_id").alias("product_id"),
    F.col("item.quantity").cast("int").alias("quantity"),
    F.col("item.unit_price").cast(DecimalType(10,2)).alias("unit_price"),
    F.col("item.discount_amount").cast(DecimalType(10,2)).alias("discount_amount"),
    (F.col("item.quantity") * F.col("item.unit_price")).cast(DecimalType(12,2)).alias("gross_amount"),
    ((F.col("item.quantity") * F.col("item.unit_price") - F.col("item.discount_amount")).cast(DecimalType(12,2))).alias("net_amount"),
    "updated_at"
)
order_items_table = DeltaTable.forName(spark, f"{catalog}.silver.order_items")
(order_items_table.alias("target")
    .merge(order_items.alias("source"), "target.order_id = source.order_id AND target.line_number = source.line_number")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

# Re-merge silver.quarantine_orders with same data
quarantine_orders_out = quarantine_orders.select(
    "order_id", "reject_reason", "updated_at",
    F.to_json(F.struct([F.col(c) for c in quarantine_orders.columns])).alias("raw_record"),
    F.current_timestamp().alias("quarantined_at")
)
quarantine_table = DeltaTable.forName(spark, f"{catalog}.silver.quarantine_orders")
(quarantine_table.alias("target")
    .merge(quarantine_orders_out.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdate(set={"reject_reason": "source.reject_reason", "updated_at": "source.updated_at", "raw_record": "source.raw_record"})
    .whenNotMatchedInsert(values={"order_id": "source.order_id", "reject_reason": "source.reject_reason", "updated_at": "source.updated_at", "raw_record": "source.raw_record", "quarantined_at": "source.quarantined_at"})
    .execute())

# Verify no changes after re-merge
orders_count_after = spark.table(f"{catalog}.silver.orders").count()
items_count_after = spark.table(f"{catalog}.silver.order_items").count()
quarantine_count_after = spark.table(f"{catalog}.silver.quarantine_orders").count()
orders_hash_after = spark.sql(f"SELECT hash(*) FROM {catalog}.silver.orders ORDER BY order_id").collect()
items_hash_after = spark.sql(f"SELECT hash(*) FROM {catalog}.silver.order_items ORDER BY order_id, line_number").collect()
quarantine_pairs_after = spark.sql(f"SELECT order_id, reject_reason FROM {catalog}.silver.quarantine_orders ORDER BY order_id").collect()

print("=== C5: Idempotency Verification ===")
print(f"silver.orders:           {orders_count_before} -> {orders_count_after}  (unchanged: {orders_count_before == orders_count_after}, values: {orders_hash_before == orders_hash_after})")
print(f"silver.order_items:      {items_count_before} -> {items_count_after}  (unchanged: {items_count_before == items_count_after}, values: {items_hash_before == items_hash_after})")
print(f"silver.quarantine_orders: {quarantine_count_before} -> {quarantine_count_after}  (pairs unchanged: {quarantine_pairs_before == quarantine_pairs_after})")